In [ ]:
!pip install qiskit qiskit-machine-learning qiskit-algorithms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.6 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. IMPORT & SETUP
# ==========================================
import pandas as pd
import numpy as np
import time
import gc
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_squared_error

# Qiskit
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ==========================================
# 2. LOAD DATA & PREPROCESSING
# ==========================================
from google.colab import files
uploaded = files.upload()
nama_file = list(uploaded.keys())[0]
df = pd.read_csv(nama_file)

# Kolom SO2, O3, NO2, dan HC sudah dibuang
feature_cols = [
    'PM10', 'CO', 'pm2.5_lag1', 'pm10_lag1',
    'pm2.5_trend3', 'pm10_trend3', 'suhu_avg',
    'curah_hujan', 'angin_max'
]

X = df[feature_cols].values
y = df['PM2.5'].values

# Split Data 80% Train, 20% Test tanpa scaler tambahan
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Data Train: {len(X_train)} hari | Data Test: {len(X_test)} hari")

Saving Data_Udara_Pekanbaru_Normalisasi.csv to Data_Udara_Pekanbaru_Normalisasi.csv
Data Train: 583 hari | Data Test: 146 hari


In [ ]:
# ==========================================
# 3. HYPERPARAMETER TUNING
# ==========================================
print("\nMULAI TUNING MODEL...")

# --- A. LIGHTGBM ---
print("   [1/2] Menyiapkan LightGBM (Default Mode)...")
best_lgbm = lgb.LGBMRegressor(random_state=42, verbose=-1)
best_lgbm.fit(X_train, y_train)

# --- B. QUANTUM SVR ---
print("   [2/2] Menyiapkan Quantum Kernel...")
# Indeks baru: pm2.5_lag1 (2), PM10 (0), pm2.5_trend3 (4), pm10_lag1 (3)
q_indices = [2, 0, 4, 3]
X_train_q = X_train[:, q_indices]
X_test_q = X_test[:, q_indices]

feature_map = ZZFeatureMap(feature_dimension=len(q_indices), reps=1, entanglement='linear')
sampler = StatevectorSampler()
fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

# Tuning sampel kecil untuk C dan Epsilon QSVR
X_tune, y_tune = resample(X_train_q, y_train, n_samples=100, random_state=42)
kernel_matrix_tune = quantum_kernel.evaluate(x_vec=X_tune, y_vec=X_tune)

svr_search = RandomizedSearchCV(
    SVR(kernel='precomputed'),
    param_distributions={'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1, 0.2]},
    n_iter=5, cv=3, scoring='neg_mean_squared_error', random_state=42
)
svr_search.fit(kernel_matrix_tune, y_tune)
best_svr_params = svr_search.best_params_


MULAI TUNING MODEL...
   [1/2] Menyiapkan LightGBM (Default Mode)...
   [2/2] Menyiapkan Quantum Kernel...


In [ ]:
# ==========================================
# EKSTRAKSI DATA SUPPORT VECTOR DENGAN TANGGAL
# ==========================================
import pandas as pd
from google.colab import files
from sklearn.utils import resample

print("Menyiapkan ulang model Q-1 untuk ekstraksi data CSV berserta tanggal...")

# A. Mengambil deret tanggal khusus untuk porsi data latih dari dataframe asli
nama_kolom_waktu = 'Tanggal'
deret_waktu_train = df[nama_kolom_waktu].iloc[:len(y_train)].values

# 1. Mengambil subsampel yang identik, sekaligus mengacak array tanggalnya
X_sub_0, y_sub_0, waktu_sub_0 = resample(X_train_q, y_train, deret_waktu_train, n_samples=150, random_state=0)

# 2. Menghitung ulang matriks kernel khusus untuk iterasi ini
train_kernel_0 = quantum_kernel.evaluate(x_vec=X_sub_0, y_vec=X_sub_0)
eval_test_kernel_0 = quantum_kernel.evaluate(x_vec=X_test_q, y_vec=X_sub_0)

# 3. Melatih ulang model SVR
qsvr_0 = SVR(kernel='precomputed', C=best_svr_params['C'], epsilon=best_svr_params['epsilon'])
qsvr_0.fit(train_kernel_0, y_sub_0)

# 4. Mengekstraksi parameter yang dibutuhkan
alphas_0 = qsvr_0.dual_coef_[0]
sv_indices_0 = qsvr_0.support_
k_values_0 = eval_test_kernel_0[0, sv_indices_0]

fitur_sv_0 = X_sub_0[sv_indices_0]
target_sv_0 = y_sub_0[sv_indices_0]

# B. Mengekstrak tanggal asli yang sesuai dengan indeks Support Vector terpilih
tanggal_sv_0 = waktu_sub_0[sv_indices_0]

# 5. Menyusun format tabel DataFrame
df_sv = pd.DataFrame({
    'Data_SV': [f"SV_{j+1}" for j in range(len(sv_indices_0))],
    'Waktu_Observasi': tanggal_sv_0,
    'Nilai_Alpha': alphas_0,
    'Nilai_Kernel_K': k_values_0,
    'Target_PM2.5_Aktual': target_sv_0,
    'pm2.5_lag1': fitur_sv_0[:, 0],
    'PM10': fitur_sv_0[:, 1],
    'pm2.5_trend3': fitur_sv_0[:, 2],
    'pm10_lag1': fitur_sv_0[:, 3]
})

# 6. Menyimpan dan mengunduh file
nama_file_csv = 'Data_Support_Vector_Sirkuit_1_Dengan_Tanggal.csv'
df_sv.to_csv(nama_file_csv, index=False)

files.download(nama_file_csv)
print(f"File '{nama_file_csv}' berhasil diunduh. Silakan buka di Excel.")

Menyiapkan ulang model Q-1 untuk ekstraksi data CSV berserta tanggal...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'Data_Support_Vector_Sirkuit_1_Dengan_Tanggal.csv' berhasil diunduh. Silakan buka di Excel.


In [ ]:
# ==========================================
# 4. TRAINING FINAL BASE MODELS
# ==========================================
print("\nMULAI TRAINING FINAL BASE MODELS...")

# 1. LightGBM
pred_train_lgbm = best_lgbm.predict(X_train)
pred_test_lgbm = best_lgbm.predict(X_test)

# 2. Quantum Bagging
print("   Menjalankan Quantum Bagging...")
n_estimators = 5
sample_size = 150

quantum_preds_train = []
quantum_preds_test = []

for i in range(n_estimators):
    print(f"      Memulai Model Q-{i+1}/{n_estimators}...")
    X_sub, y_sub = resample(X_train_q, y_train, n_samples=sample_size, random_state=i*42)

    train_kernel = quantum_kernel.evaluate(x_vec=X_sub, y_vec=X_sub)
    eval_train_kernel = quantum_kernel.evaluate(x_vec=X_train_q, y_vec=X_sub)
    eval_test_kernel = quantum_kernel.evaluate(x_vec=X_test_q, y_vec=X_sub)

    qsvr = SVR(kernel='precomputed', C=best_svr_params['C'], epsilon=best_svr_params['epsilon'])
    qsvr.fit(train_kernel, y_sub)

    quantum_preds_train.append(qsvr.predict(eval_train_kernel))
    quantum_preds_test.append(qsvr.predict(eval_test_kernel))

    # =====================================================================
    # RUMUS SKRIPSI
    # =====================================================================
    if i == 0:
        print("\n" + "="*60)
        print("BONGKAR RUMUS PREDIKSI MANUAL QSVR (Sirkuit 1)")
        print("="*60)

        bias = qsvr.intercept_[0]
        alphas = qsvr.dual_coef_[0]
        sv_indices = qsvr.support_

        print(f"Nilai Bias b = {bias:.4f}".replace('.', ','))
        print(f"Jumlah Support Vector = {len(sv_indices)} data")

        print("\nContoh 3 Nilai Alpha Pertama:")
        for j in range(min(3, len(alphas))):
            print(f"Alpha_{j+1} = {alphas[j]:.4f}".replace('.', ','))

        k_values = eval_test_kernel[0, sv_indices]

        print("\nContoh 3 Nilai Kernel K (Data Uji ke-1 dengan Support Vectors):")
        for j in range(min(3, len(k_values))):
             print(f"K(sv_{j+1}, x_test_1) = {k_values[j]:.4f}".replace('.', ','))

        prediksi_manual = np.sum(alphas * k_values) + bias
        prediksi_mesin = quantum_preds_test[0][0]

        print(f"\n1. Penjumlahan Alpha * K = {np.sum(alphas * k_values):.4f}".replace('.', ','))
        print(f"2. Ditambah Bias b = {bias:.4f}".replace('.', ','))
        print(f"3. HASIL TEBAKAN MANUAL = {prediksi_manual:.4f}".replace('.', ','))
        print(f"4. HASIL TEBAKAN MESIN = {prediksi_mesin:.4f}".replace('.', ','))
        print("="*60 + "\n")
    # =====================================================================

    del qsvr, train_kernel, eval_train_kernel, eval_test_kernel; gc.collect()

pred_train_qsvr = np.mean(quantum_preds_train, axis=0)
pred_test_qsvr = np.mean(quantum_preds_test, axis=0)
print("Prediksi Kuantum Selesai!")



MULAI TRAINING FINAL BASE MODELS...
   Menjalankan Quantum Bagging...
      Memulai Model Q-1/5...

BONGKAR RUMUS PREDIKSI MANUAL QSVR (Sirkuit 1)
Nilai Bias b = 0,4025
Jumlah Support Vector = 30 data

Contoh 3 Nilai Alpha Pertama:
Alpha_1 = -0,2617
Alpha_2 = 0,5998
Alpha_3 = 0,0554

Contoh 3 Nilai Kernel K (Data Uji ke-1 dengan Support Vectors):
K(sv_1, x_test_1) = 0,6543
K(sv_2, x_test_1) = 0,3301
K(sv_3, x_test_1) = 0,1328

1, Penjumlahan Alpha * K = -0,2067
2, Ditambah Bias b = 0,4025
3, HASIL TEBAKAN MANUAL = 0,1958
4, HASIL TEBAKAN MESIN = 0,1958

      Memulai Model Q-2/5...
      Memulai Model Q-3/5...
      Memulai Model Q-4/5...
      Memulai Model Q-5/5...
Prediksi Kuantum Selesai!


In [ ]:
# ==========================================
# 5. META-LEARNER STACKING & EVALUASI AKHIR
# ==========================================
print("\nMELATIH META-LEARNER DAN EVALUASI AKHIR...")

X_meta_train = np.column_stack((pred_train_lgbm, pred_train_qsvr))
X_meta_test = np.column_stack((pred_test_lgbm, pred_test_qsvr))

meta_model = Ridge(alpha=1.0, positive=True, fit_intercept=True)
meta_model.fit(X_meta_train, y_train)

weights = meta_model.coef_ / np.sum(meta_model.coef_)
bias_meta = meta_model.intercept_
print(f"   LightGBM Weight : {weights[0]:.4f}")
print(f"   Quantum Weight  : {weights[1]:.4f}")
print(f"   Meta Bias       : {bias_meta:.4f}")

final_pred = meta_model.predict(X_meta_test)

r2_hybrid = r2_score(y_test, final_pred)
r2_lgbm = r2_score(y_test, pred_test_lgbm)

rmse_hybrid = np.sqrt(mean_squared_error(y_test, final_pred))
rmse_lgbm = np.sqrt(mean_squared_error(y_test, pred_test_lgbm))

print("\n" + "="*50)
print("HASIL AKHIR PENGUJIAN KESELURUHAN (Skala Normalisasi 0-1)")
print("="*50)
print(f"LightGBM Tunggal -> R2: {r2_lgbm:.4f} | RMSE: {rmse_lgbm:.4f}")
print(f"Hibrida Ansambel -> R2: {r2_hybrid:.4f} | RMSE: {rmse_hybrid:.4f}")
print("="*50)


MELATIH META-LEARNER DAN EVALUASI AKHIR...
   LightGBM Weight : 0.7649
   Quantum Weight  : 0.2351
   Meta Bias       : -0.0105

HASIL AKHIR PENGUJIAN KESELURUHAN (Skala Normalisasi 0-1)
LightGBM Tunggal -> R2: 0.7748 | RMSE: 0.0387
Hibrida Ansambel -> R2: 0.7779 | RMSE: 0.0384


In [ ]:
# ==========================================
# EKSTRAKSI TABEL PREDIKSI META-LEARNER
# ==========================================
import pandas as pd
from google.colab import files

print("Menyiapkan tabel hasil prediksi untuk Meta-Learner...")

# Memastikan jumlah data sama
jumlah_data_uji = len(y_test)

# Membuat DataFrame untuk menampung hasil prediksi
df_prediksi = pd.DataFrame({
    'Data_Uji_ke': [i+1 for i in range(jumlah_data_uji)],
    'Target_Aktual_PM2.5': y_test,
    'Tebakan_LightGBM': pred_test_lgbm,
    'Tebakan_QSVR': pred_test_qsvr
})

# Menyimpan dan mengunduh file
nama_file_prediksi = 'Tabel_Input_Meta_Learner.csv'
df_prediksi.to_csv(nama_file_prediksi, index=False)

files.download(nama_file_prediksi)
print(f"File '{nama_file_prediksi}' berhasil diunduh.")

Menyiapkan tabel hasil prediksi untuk Meta-Learner...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'Tabel_Input_Meta_Learner.csv' berhasil diunduh.


In [ ]:
# ==========================================
# EKSTRAKSI TABEL PREDIKSI LIGHTGBM TUNGGAL
# ==========================================
import pandas as pd
from google.colab import files

print("Menyiapkan tabel komparasi nilai aktual dan prediksi LightGBM...")

# Menyusun format tabel DataFrame
df_lgbm_pred = pd.DataFrame({
    'Indeks_Data_Uji': [i+1 for i in range(len(y_test))],
    'Target_Aktual_PM2.5': y_test,
    'Tebakan_LightGBM': pred_test_lgbm
})

# Menyimpan dan mengunduh file otomatis
nama_file = 'Tabel_Prediksi_LightGBM.csv'
df_lgbm_pred.to_csv(nama_file, index=False)

from google.colab import files
files.download(nama_file)
print(f"File {nama_file} berhasil diunduh. Silakan buka di Excel.")

Menyiapkan tabel komparasi nilai aktual dan prediksi LightGBM...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File Tabel_Prediksi_LightGBM.csv berhasil diunduh. Silakan buka di Excel.


part 2

In [ ]:
# ==========================================
# EKSTRAKSI TABEL PREDIKSI ARSITEKTUR KUANTUM
# ==========================================
import pandas as pd
from google.colab import files

print("Menyiapkan tabel komparasi nilai aktual dan prediksi arsitektur kuantum...")

# Mengambil deret waktu khusus untuk porsi data uji
nama_kolom_waktu = 'Tanggal'
deret_waktu_uji = df[nama_kolom_waktu].iloc[-len(y_test):].values

# Menyusun format tabel DataFrame
df_qsvr_pred = pd.DataFrame({
    'Indeks_Data_Uji': [i+1 for i in range(len(y_test))],
    'Waktu_Observasi': deret_waktu_uji,
    'Target_Aktual_PM2.5': y_test,
    'Prediksi_Arsitektur_Kuantum': pred_test_qsvr
})

# Menyimpan dan mengunduh file
nama_file_qsvr = 'Tabel_Prediksi_Kuantum_Final.csv'
df_qsvr_pred.to_csv(nama_file_qsvr, index=False)

files.download(nama_file_qsvr)
print(f"File '{nama_file_qsvr}' berhasil diunduh. Silakan buka di Excel.")

Menyiapkan tabel komparasi nilai aktual dan prediksi arsitektur kuantum...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'Tabel_Prediksi_Kuantum_Final.csv' berhasil diunduh. Silakan buka di Excel.


In [ ]:
# ==========================================
# EKSTRAKSI TABEL 4.13 KOMPARASI FINAL HIBRIDA
# ==========================================
import pandas as pd
from google.colab import files

print("Menyiapkan Tabel 4.13 Komparasi Keluaran Prediksi Final...")

# Mengambil deret waktu khusus untuk porsi data uji
# Ganti 'Tanggal' dengan nama kolom waktu di dataset asli Anda jika berbeda
nama_kolom_waktu = 'Tanggal'
deret_waktu_uji = df[nama_kolom_waktu].iloc[-len(y_test):].values

# Menyusun format tabel DataFrame
df_tabel_413 = pd.DataFrame({
    'Waktu_Observasi': deret_waktu_uji,
    'Target_Aktual_PM2.5': y_test,
    'Prediksi_Dasar_LightGBM': pred_test_lgbm,
    'Prediksi_Dasar_Kuantum': pred_test_qsvr,
    'Prediksi_Akhir_Hibrida': final_pred
})

# Menyimpan dan mengunduh file
nama_file_413 = 'Tabel_4_13_Komparasi_Final.csv'
df_tabel_413.to_csv(nama_file_413, index=False)

files.download(nama_file_413)
print(f"File {nama_file_413} berhasil diunduh. Silakan buka di perangkat lunak hamparan data.")

Menyiapkan Tabel 4.13 Komparasi Keluaran Prediksi Final...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File Tabel_4_13_Komparasi_Final.csv berhasil diunduh. Silakan buka di perangkat lunak hamparan data.


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Menghitung RMSE dan R2 untuk LightGBM
rmse_lgbm = np.sqrt(mean_squared_error(y_test, pred_test_lgbm))
r2_lgbm = r2_score(y_test, pred_test_lgbm)

# Menghitung RMSE dan R2 untuk QSVR
rmse_qsvr = np.sqrt(mean_squared_error(y_test, pred_test_qsvr))
r2_qsvr = r2_score(y_test, pred_test_qsvr)

print("=== NILAI UNTUK TABEL 4.14 ===")
print(f"RMSE LightGBM : {rmse_lgbm:.4f} | R2 LightGBM : {r2_lgbm:.4f}")
print(f"RMSE QSVR     : {rmse_qsvr:.4f} | R2 QSVR     : {r2_qsvr:.4f}")

=== NILAI UNTUK TABEL 4.14 ===
RMSE LightGBM : 0.0387 | R2 LightGBM : 0.7748
RMSE QSVR     : 0.0651 | R2 QSVR     : 0.3612


In [ ]:
import numpy as np

print(f"Nilai n (Jumlah Data Uji) = {len(y_test)}")
print(f"Nilai y-bar (Rata-Rata Aktual) = {np.mean(y_test):.4f}")

Nilai n (Jumlah Data Uji) = 146
Nilai y-bar (Rata-Rata Aktual) = 0.1496
